# Session 2: Complete Clustering Analysis, Text Mining & Temporal Exploration

**Objectives:**
1. ✅ Optimize and compare clustering algorithms (DBSCAN, K-Means, HDBSCAN)
2. ✅ Implement text mining for automatic cluster naming (TF-IDF + Keywords)
3. ✅ Explore temporal patterns in Flickr data
4. ✅ Prepare final demo with integrated cluster names

**Data Source:** Flickr photos from Lyon, France with GPS coordinates and metadata

## Section 1: Setup & Load Data

In [ ]:
# Import required libraries
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, '../src')

# Set style for better visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

print("✅ Libraries loaded successfully")

In [ ]:
# Load and clean data
from load_data import load_data, print_report
from cleaning import clean_data, print_cleaning_report

print("Loading raw data...")
df_raw, rep_raw = load_data("../flickr_data2.csv")
print(f"Loaded {len(df_raw):,} photos")

print("\nCleaning data...")
df_clean, rep_clean = clean_data(df_raw)
print(f"Cleaned data: {len(df_clean):,} photos")
print_cleaning_report(rep_clean)

## Section 2: Optimize & Compare Clustering Algorithms

In [ ]:
# Run algorithm comparison
from comparison import (
    compare_algorithms,
    print_comparison_table,
    plot_elbow_silhouette,
)

print("="*80)
print("COMPARING 3 CLUSTERING ALGORITHMS")
print("="*80)

comparison_df = compare_algorithms(
    df_clean,
    dbscan_params={"eps_meters": 50.0, "min_samples": 50, "deduplicate_coords": True},
    kmeans_params={"n_clusters": 50},
    hdbscan_params={"min_cluster_size": 50, "min_samples": 50},
)

print_comparison_table(comparison_df)
comparison_df

In [ ]:
# Plot K-Means optimization curves
print("\nTesting K-Means for optimal K value...")
try:
    plot_elbow_silhouette(df_clean, k_range=range(20, 81, 10))
    print("✅ Elbow plot generated successfully")
except Exception as e:
    print(f"⚠️ Could not generate elbow plot: {e}")

## Section 3: Algorithm Discussion & Recommendation

### Analysis

**Key Observations:**

1. **DBSCAN** (Recommended):
   - ✅ Automatically discovers number of clusters (~49 clusters)
   - ✅ Handles noise points (outliers in sparse regions)
   - ✅ No need to specify K in advance
   - ✅ Clear parameter interpretations (eps=50m = 1 city block)
   - ✅ Matches real Lyon geography

2. **K-Means**:
   - ✅ Fast computation and simple interpretation
   - ❌ Requires choosing K beforehand
   - ❌ Forces all points into clusters (no noise handling)
   - ❌ Assumes spherical clusters (not ideal for POI shapes)

3. **HDBSCAN**:
   - ✅ Handles variable density zones
   - ✅ Hierarchical structure
   - ❌ More complex parameters
   - ❌ Longer computation time

### Recommendation

**Use DBSCAN for discovering POIs in urban areas** because:
1. Urban photography has clustered hotspots (natural POIs) vs. sparse residential areas
2. eps=50m parameter has clear meaning (dense city block)
3. Automatic cluster discovery suits exploratory analysis
4. Results align with real Lyon geography (Bellecour, Old Town, Park, etc.)

## Section 4: Apply Optimal Clustering (DBSCAN)

In [ ]:
# Run DBSCAN clustering with optimal parameters
from clustering import run_dbscan_geo, print_cluster_report, save_clustered_csv

print("Running DBSCAN with optimal parameters...")
print("  - eps: 50 meters (dense city block)")
print("  - min_samples: 50 photos (significant POI)")
print("  - deduplicate_coords: True (avoid mega-clusters)")

df_clustered, rep_cluster = run_dbscan_geo(
    df_clean,
    eps_meters=50.0,
    min_samples=50,
    deduplicate_coords=True,
    coord_precision=4,
)

print_cluster_report(rep_cluster)

# Save clustered data
os.makedirs("../outputs", exist_ok=True)
out_csv = save_clustered_csv(df_clustered, "../outputs/clustered.csv")
print(f"\n✅ Clustered data saved to: {out_csv}")

## Section 5: Text Mining for Cluster Naming (TF-IDF)

In [ ]:
# Preprocess text and extract cluster descriptions
from text_mining import (
    preprocess_text,
    extract_cluster_descriptions,
    save_descriptions_csv,
    print_cluster_descriptions,
)

print("Preprocessing text data...")
df_clustered = preprocess_text(df_clustered, text_col="text")
print(f"✅ Text preprocessed: {df_clustered['text'].notna().sum()} rows with text")

print("\nExtracting cluster descriptions using TF-IDF...")
descriptions = extract_cluster_descriptions(
    df_clustered,
    cluster_col="cluster",
    text_col="text",
    top_n_keywords=10,
    min_df=2,
    max_df=0.8,
)

print(f"✅ Extracted {len(descriptions)} cluster descriptions")

# Print top descriptions
print_cluster_descriptions(descriptions, top_n=15)

# Save descriptions to CSV
out_desc_csv = save_descriptions_csv(descriptions, "../outputs/cluster_descriptions.csv")
print(f"\n✅ Descriptions saved to: {out_desc_csv}")

In [ ]:
# Create word clouds for top clusters
from text_mining import create_wordcloud_for_cluster

print("\nGenerating word clouds for top 5 clusters...\n")
for i, desc in enumerate(descriptions[:5]):
    try:
        out_wordcloud = create_wordcloud_for_cluster(
            df_clustered,
            desc.cluster_id,
            output_path=f"../outputs/wordcloud_cluster_{desc.cluster_id}.png",
        )
        if out_wordcloud:
            print(f"✅ [{i+1}] Cluster {desc.cluster_id}: {out_wordcloud}")
    except Exception as e:
        print(f"⚠️ Could not create wordcloud for cluster {desc.cluster_id}: {e}")

## Section 6: Temporal Exploration & Analysis

In [ ]:
# Parse datetime column
print("Preparing temporal analysis...")
df_clustered["taken_dt"] = pd.to_datetime(df_clustered["taken_dt"], errors="coerce")
df_temporal = df_clustered[df_clustered["taken_dt"].notna()].copy()

print(f"✅ Temporal data: {len(df_temporal):,} photos with valid dates")
print(f"   Date range: {df_temporal['taken_dt'].min().date()} to {df_temporal['taken_dt'].max().date()}")
print(f"   Total span: {(df_temporal['taken_dt'].max() - df_temporal['taken_dt'].min()).days} days")

# Monthly analysis
df_temporal["year_month"] = df_temporal["taken_dt"].dt.to_period("M")
monthly_counts = df_temporal.groupby("year_month").size()

print(f"\n📊 Monthly Statistics:")
print(f"   Peak month: {monthly_counts.idxmax()} with {monthly_counts.max():,} photos")
print(f"   Average per month: {monthly_counts.mean():.0f} photos")
print(f"   Median per month: {monthly_counts.median():.0f} photos")

In [ ]:
# Time series visualization - Photos per month
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

# Line plot: Photos per month
monthly_counts.plot(ax=axes[0], marker='o', linewidth=2, markersize=6, color='steelblue')
axes[0].set_title('Flickr Photos Over Time (Monthly)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Time Period')
axes[0].set_ylabel('Number of Photos')
axes[0].grid(True, alpha=0.3)
axes[0].tick_params(axis='x', rotation=45)

# Cumulative plot: Total photos over time
cumulative = monthly_counts.cumsum()
cumulative.plot(ax=axes[1], marker='o', linewidth=2, markersize=6, color='darkgreen')
axes[1].set_title('Cumulative Flickr Photos Over Time', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time Period')
axes[1].set_ylabel('Cumulative Number of Photos')
axes[1].grid(True, alpha=0.3)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../outputs/temporal_timeline.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Temporal timeline saved to: ../outputs/temporal_timeline.png")

In [ ]:
# Cluster activity over time
print("Analyzing cluster activity over time...\n")

# Get top clusters by size
top_clusters = df_clustered[df_clustered["cluster"] != -1]["cluster"].value_counts().head(10).index.tolist()

# Create heatmap data: cluster x month
cluster_temporal = df_temporal[df_temporal["cluster"].isin(top_clusters)].copy()
pivot_data = cluster_temporal.groupby(["cluster", "year_month"]).size().unstack(fill_value=0)

# Plot heatmap
fig, ax = plt.subplots(figsize=(16, 8))
sns.heatmap(pivot_data, cmap="YlOrRd", ax=ax, cbar_kws={"label": "Number of Photos"})
ax.set_title('Top 10 Clusters: Activity Over Time (Heatmap)', fontsize=14, fontweight='bold')
ax.set_xlabel('Time Period')
ax.set_ylabel('Cluster ID')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../outputs/cluster_temporal_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print("✅ Temporal heatmap saved to: ../outputs/cluster_temporal_heatmap.png")

In [ ]:
# Cluster temporal statistics
print("Top 10 Clusters by Temporal Span:\n")

cluster_stats = []
for cluster_id in df_clustered[df_clustered["cluster"] != -1]["cluster"].unique():
    cluster_df = df_clustered[df_clustered["cluster"] == cluster_id]
    temporal_cluster = cluster_df[cluster_df["taken_dt"].notna()]
    
    if len(temporal_cluster) > 0:
        min_date = temporal_cluster["taken_dt"].min()
        max_date = temporal_cluster["taken_dt"].max()
        span_days = (max_date - min_date).days
        n_photos = len(cluster_df)
        
        # Get cluster description
        cluster_desc = next((d for d in descriptions if d.cluster_id == cluster_id), None)
        desc_text = cluster_desc.description if cluster_desc else f"Cluster {cluster_id}"
        
        cluster_stats.append({
            "Cluster ID": int(cluster_id),
            "Description": desc_text,
            "Photos": n_photos,
            "First Photo": min_date.date(),
            "Last Photo": max_date.date(),
            "Span (days)": span_days,
        })

cluster_stats_df = pd.DataFrame(cluster_stats).sort_values("Photos", ascending=False).head(10)
print(cluster_stats_df.to_string(index=False))

# Save to CSV
cluster_stats_df.to_csv("../outputs/cluster_temporal_stats.csv", index=False)
print("\n✅ Temporal statistics saved to: ../outputs/cluster_temporal_stats.csv")

## Section 7: Create Enhanced Cluster Map with Names

In [ ]:
# Create enhanced map with cluster names
from visualization import create_cluster_map_with_names

print("Creating enhanced cluster map with TF-IDF names...\n")

try:
    out_map = create_cluster_map_with_names(
        df_clustered,
        descriptions=descriptions,
        output_html="../outputs/map_clusters_named.html",
        sample_n=25000,
    )
    print(f"✅ Enhanced cluster map saved to: {out_map}")
    print("\n📍 Map features:")
    print("   - Colored points by cluster")
    print("   - Cluster centers marked with icons")
    print("   - Cluster names from TF-IDF analysis")
    print("   - Interactive popups with cluster info")
except Exception as e:
    print(f"⚠️ Could not create cluster map: {e}")
    import traceback
    traceback.print_exc()

## Section 8: Summary & Results

In [ ]:
# Generate comprehensive summary
print("="*80)
print("📊 SESSION 2 COMPLETE ANALYSIS - SUMMARY")
print("="*80)

print("\n1️⃣ ALGORITHM COMPARISON:")
print(f"   ✅ DBSCAN (Recommended)")
print(f"      - Clusters: {rep_cluster.n_clusters}")
print(f"      - Noise: {rep_cluster.noise_points:,} ({rep_cluster.noise_ratio*100:.1f}%)")
print(f"      - Largest cluster: {rep_cluster.cluster_sizes_top10[0][1]:,} photos")
print(f"      - Parameters: eps=50m, min_samples=50")

print("\n2️⃣ TEXT MINING RESULTS:")
print(f"   ✅ Extracted {len(descriptions)} cluster descriptions")
print(f"   ✅ Using TF-IDF with bigrams (1-2 word terms)")
print(f"   ✅ Generated word clouds for top 5 clusters")

print("\n3️⃣ TEMPORAL ANALYSIS:")
print(f"   ✅ Data span: {(df_temporal['taken_dt'].max() - df_temporal['taken_dt'].min()).days} days")
print(f"   ✅ Peak month: {monthly_counts.idxmax()}")
print(f"   ✅ Total photos with dates: {len(df_temporal):,}")

print("\n4️⃣ GENERATED FILES:")
files_created = [
    ("Clustered data", "../outputs/clustered.csv"),
    ("Algorithm comparison", "../outputs/comparison_metrics.csv"),
    ("Cluster descriptions", "../outputs/cluster_descriptions.csv"),
    ("Temporal stats", "../outputs/cluster_temporal_stats.csv"),
    ("Temporal timeline", "../outputs/temporal_timeline.png"),
    ("Temporal heatmap", "../outputs/cluster_temporal_heatmap.png"),
    ("Enhanced map", "../outputs/map_clusters_named.html"),
    ("Word clouds", "../outputs/wordcloud_cluster_*.png"),
]

for i, (name, path) in enumerate(files_created, 1):
    print(f"   [{i}] {name:25s} -> {path}")

print("\n" + "="*80)
print("✅ SESSION 2 ANALYSIS COMPLETE - READY FOR DEMO!")
print("="*80)